## 1. Import Libraries and Setup


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, LabelEncoder

## 2. Data Loading


In [57]:
# Configuration
MERGED_DATA_DIR = '../../data/merged'
FINAL_DATA_DIR = '../../data/final'

# Ensure output directory exists
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

print("=" * 80)
print("Merged Dataset Preprocessing Pipeline")
print("=" * 80)
print(f"\nMerged data location: {MERGED_DATA_DIR}")
print(f"Final location: {FINAL_DATA_DIR}\n")

Merged Dataset Preprocessing Pipeline

Merged data location: ../../data/merged
Final location: ../../data/final



In [58]:
MERGED_FILE = os.path.join(MERGED_DATA_DIR, 'merged.csv')
FINAL_FILE = os.path.join(FINAL_DATA_DIR, 'encoded_unsupervised.csv')

In [59]:
# Load merged dataset
df = pd.read_csv(MERGED_FILE)

# Display initial info
print("=" * 80)
print("DATAFRAME BEFORE ENCODING")
print("=" * 80)

print(f"\nShape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

DATAFRAME BEFORE ENCODING

Shape: (32548, 21)

Columns (21): ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'date_unregistration', 'total_clicks', 'num_sites', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay']

Data types:
code_module                       str
code_presentation                 str
id_student                      int64
gender                            str
region                            str
highest_education                 str
imd_band                          str
age_band                          str
num_of_prev_attempts            int64
studied_credits                 int64
disability                        str
final_result                      str
date_registration             float64
date_unregistration           float64
total

## 3. Clean and Prepare Data

Drop id_student, date_unregistration, and is_withdrawn to remove data leakage and PII.


In [60]:
# Drop columns that cause data leakage and PII
columns_to_drop = ['id_student', 'date_unregistration', 'region', 'num_sites', 'total_clicks']
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df_encoded = df.drop(columns=columns_to_drop)

print("\n" + "=" * 80)
print("STEP 1: DATA CLEANING")
print("=" * 80)
print(f"\nColumns dropped (data leakage + PII): {columns_to_drop}")
print(f"Shape after dropping: {df_encoded.shape}")
print(f"Remaining columns: {df_encoded.columns.tolist()}")


STEP 1: DATA CLEANING

Columns dropped (data leakage + PII): ['id_student', 'date_unregistration', 'region', 'num_sites', 'total_clicks']
Shape after dropping: (32548, 16)
Remaining columns: ['code_module', 'code_presentation', 'gender', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay']


## 4. Apply Ordinal Mappings

Map educational levels, age bands, and socioeconomic indices to ordinal values.


In [61]:
print("\n" + "=" * 80)
print("STEP 2: ORDINAL MAPPINGS")
print("=" * 80)

# 1. Highest Education Mapping
education_mapping = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}

if 'highest_education' in df_encoded.columns:
    print(f"\nBefore education mapping:")
    print(df_encoded['highest_education'].value_counts())
    
    df_encoded['highest_education'] = df_encoded['highest_education'].map(education_mapping)
    print(f"After education mapping:")
    print(df_encoded['highest_education'].value_counts(dropna=False))



STEP 2: ORDINAL MAPPINGS

Before education mapping:
highest_education
A Level or Equivalent          14026
Lower Than A Level             13138
HE Qualification                4725
No Formal quals                  346
Post Graduate Qualification      313
Name: count, dtype: int64
After education mapping:
highest_education
2    14026
1    13138
3     4725
0      346
4      313
Name: count, dtype: int64


In [62]:
# 2. Age Band Mapping
age_mapping = {
    '0-35': 0,
    '35-55': 1,
    '55+': 2
}

if 'age_band' in df_encoded.columns:
    print(f"\nBefore age_band mapping:")
    print(df_encoded['age_band'].value_counts())
    
    df_encoded['age_band'] = df_encoded['age_band'].map(age_mapping)
    print(f"After age_band mapping:")
    print(df_encoded['age_band'].value_counts(dropna=False))



Before age_band mapping:
age_band
0-35     22911
35-55     9423
55+        214
Name: count, dtype: int64
After age_band mapping:
age_band
0    22911
1     9423
2      214
Name: count, dtype: int64


In [63]:
# 3. IMD Band Mapping with region-based imputation for 'None'
imd_mapping = {
    '0-10%': 0,
    '10-20%': 1,
    '20-30%': 2,
    '30-40%': 3,
    '40-50%': 4,
    '50-60%': 5,
    '60-70%': 6,
    '70-80%': 7,
    '80-90%': 8,
    '90-100%': 9
}

if 'imd_band' in df_encoded.columns:
    print(f"\nBefore imd_band mapping:")
    print(df_encoded['imd_band'].value_counts(dropna=False))
    
    # Handle 'None' by imputing mode per region
    none_mask = df_encoded['imd_band'] == 'None'
    
    if none_mask.sum() > 0 and 'region' in df_encoded.columns:
        print(f"\nImputing {none_mask.sum()} 'None' values by region mode...")
        
        for region in df_encoded.loc[none_mask, 'region'].unique():
            region_mask = (df_encoded['region'] == region) & ~none_mask
            if region_mask.sum() > 0:
                mode_value = df_encoded.loc[region_mask, 'imd_band'].mode()
                if len(mode_value) > 0:
                    df_encoded.loc[(df_encoded['region'] == region) & none_mask, 'imd_band'] = mode_value[0]
    
    # Apply mapping
    df_encoded['imd_band'] = df_encoded['imd_band'].map(imd_mapping)
    print(f"After imd_band mapping:")
    print(df_encoded['imd_band'].value_counts(dropna=False))


Before imd_band mapping:
imd_band
10-20%     4242
20-30%     3651
0-10%      3618
30-40%     3541
40-50%     3250
50-60%     3130
60-70%     2899
70-80%     2877
80-90%     2758
90-100%    2582
Name: count, dtype: int64
After imd_band mapping:
imd_band
1    4242
2    3651
0    3618
3    3541
4    3250
5    3130
6    2899
7    2877
8    2758
9    2582
Name: count, dtype: int64


## 5. Perform One-Hot Encoding

Use pd.get_dummies for categorical features: code_module, code_presentation, and gender. Drop first category to avoid multicollinearity.


In [64]:
print("\n" + "=" * 80)
print("STEP 3: ONE-HOT ENCODING")
print("=" * 80)

# Identify categorical columns for one-hot encoding
categorical_cols = ['code_module', 'code_presentation', 'gender', 'disability']
categorical_cols = [col for col in categorical_cols if col in df_encoded.columns]

print(f"\nColumns to one-hot encode: {categorical_cols}")

# Apply one-hot encoding (drop first category to avoid multicollinearity)
df_encoded = pd.get_dummies(
    df_encoded,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)



STEP 3: ONE-HOT ENCODING

Columns to one-hot encode: ['code_module', 'code_presentation', 'gender', 'disability']


In [65]:

print(f"\nShape after one-hot encoding: {df_encoded.shape}")
print(f"\nNew columns created:")
new_cols = [col for col in df_encoded.columns if any(cat in col for cat in categorical_cols)]
print(new_cols)

print(f"\nTotal columns now: {len(df_encoded.columns)}")


Shape after one-hot encoding: (32548, 23)

New columns created:
['code_module_BBB', 'code_module_CCC', 'code_module_DDD', 'code_module_EEE', 'code_module_FFF', 'code_module_GGG', 'code_presentation_2013J', 'code_presentation_2014B', 'code_presentation_2014J', 'gender_M', 'disability_Y']

Total columns now: 23


## 6. Create Reliability Score

Create binary score_reliability feature: 1 if total_weight_available >= 100, else 0.


## 7. Scale Numerical Features

Apply StandardScaler to: total_clicks, studied_credits, tma_cma_weighted_score, total_submissions, avg_submission_delay.


In [66]:
print("\n" + "=" * 80)
print("STEP 5: NUMERICAL FEATURE SCALING")
print("=" * 80)

# Define feature groups with different scaling strategies
features_robust_skewed = ['engagement_intensity', 'avg_submission_delay', 'total_submissions']
features_standard_normal = ['date_registration', 'module_presentation_length']
features_robust_discrete = ['num_of_prev_attempts', 'late_submissions_count']
features_minmax_bounded = ['tma_cma_weighted_score']

# Check which features exist in the dataframe
robust_skewed_present = [col for col in features_robust_skewed if col in df_encoded.columns]
standard_normal_present = [col for col in features_standard_normal if col in df_encoded.columns]
robust_discrete_present = [col for col in features_robust_discrete if col in df_encoded.columns]
minmax_bounded_present = [col for col in features_minmax_bounded if col in df_encoded.columns]

print(f"\n--- Feature Scaling Strategy ---")
print(f"RobustScaler (Skewed): {robust_skewed_present}")
print(f"StandardScaler (Normal): {standard_normal_present}")
print(f"RobustScaler (Discrete): {robust_discrete_present}")
print(f"MinMaxScaler (Bounded): {minmax_bounded_present}")

# Apply RobustScaler to skewed features
if robust_skewed_present:
    print(f"\n--- Scaling RobustScaler (Skewed) Features ---")
    print(f"Before scaling:")
    print(df_encoded[robust_skewed_present].describe())
    
    scaler_robust_skewed = RobustScaler()
    df_encoded[robust_skewed_present] = scaler_robust_skewed.fit_transform(df_encoded[robust_skewed_present])
    
    print(f"After scaling:")
    print(df_encoded[robust_skewed_present].describe())

# Apply StandardScaler to normal features
if standard_normal_present:
    print(f"\n--- Scaling StandardScaler (Normal) Features ---")
    print(f"Before scaling:")
    print(df_encoded[standard_normal_present].describe())
    
    scaler_standard_normal = StandardScaler()
    df_encoded[standard_normal_present] = scaler_standard_normal.fit_transform(df_encoded[standard_normal_present])
    
    print(f"After scaling:")
    print(df_encoded[standard_normal_present].describe())

# Apply RobustScaler to discrete features
if robust_discrete_present:
    print(f"\n--- Scaling RobustScaler (Discrete) Features ---")
    print(f"Before scaling:")
    print(df_encoded[robust_discrete_present].describe())
    
    scaler_robust_discrete = RobustScaler()
    df_encoded[robust_discrete_present] = scaler_robust_discrete.fit_transform(df_encoded[robust_discrete_present])
    
    print(f"After scaling:")
    print(df_encoded[robust_discrete_present].describe())

# Apply MinMaxScaler to bounded features
if minmax_bounded_present:
    print(f"\n--- Scaling MinMaxScaler (Bounded) Features ---")
    print(f"Before scaling:")
    print(df_encoded[minmax_bounded_present].describe())
    
    scaler_minmax_bounded = MinMaxScaler()
    df_encoded[minmax_bounded_present] = scaler_minmax_bounded.fit_transform(df_encoded[minmax_bounded_present])
    
    print(f"After scaling (should be in [0, 1]):")
    print(df_encoded[minmax_bounded_present].describe())

if not (robust_skewed_present or standard_normal_present or robust_discrete_present or minmax_bounded_present):
    print("\nWarning: No features found to scale")


STEP 5: NUMERICAL FEATURE SCALING

--- Feature Scaling Strategy ---
RobustScaler (Skewed): ['avg_submission_delay', 'total_submissions']
StandardScaler (Normal): ['date_registration', 'module_presentation_length']
RobustScaler (Discrete): ['num_of_prev_attempts', 'late_submissions_count']
MinMaxScaler (Bounded): ['tma_cma_weighted_score']

--- Scaling RobustScaler (Skewed) Features ---
Before scaling:
       avg_submission_delay  total_submissions
count          32548.000000       32548.000000
mean              -9.703851           5.343032
std               23.872070           4.324641
min             -236.000000           0.000000
25%               -4.285714           1.000000
50%                0.000000           5.000000
75%                0.444444           9.000000
max              187.000000          14.000000
After scaling:
       avg_submission_delay  total_submissions
count          32548.000000       32548.000000
mean              -2.051485           0.042879
std            

In [67]:
df_encoded.head()

,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,final_result,date_registration,module_presentation_length,tma_cma_weighted_score,total_submissions,...,code_module_CCC,code_module_DDD,code_module_EEE,code_module_FFF,code_module_GGG,code_presentation_2013J,code_presentation_2014B,code_presentation_2014J,gender_M,disability_Y
0,3,9,2,0.0,240,Pass,-1.818699,0.909289,1.0,0.000,...,0,0,0,0,0,1,0,0,1,0
1,3,2,1,0.0,60,Pass,0.333158,0.909289,1.0,0.000,...,0,0,0,0,0,1,0,0,0,0
2,2,3,1,0.0,60,Withdrawn,-0.458563,0.909289,0.0,-0.625,...,0,0,0,0,0,1,0,0,0,1
3,2,5,1,0.0,60,Pass,0.353459,0.909289,1.0,0.000,...,0,0,0,0,0,1,0,0,0,0
4,1,5,0,0.0,60,Pass,-2.163809,0.909289,1.0,0.000,...,0,0,0,0,0,1,0,0,0,0


## 8. Export Final Encoded Dataset

Save the final encoded and scaled dataframe to data/final/final_encoded_data.csv.


In [68]:
# Remove final_result if exists
if 'final_result' in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=['final_result'])

In [69]:
print("\n" + "=" * 80)
print("STEP 6: EXPORT FINAL ENCODED DATASET")
print("=" * 80)

# Ensure output directory exists
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

# Save to data/final/
FINAL_FILE_PATH = os.path.join(FINAL_DATA_DIR, 'encoded_unsupervised.csv')
df_encoded.to_csv(FINAL_FILE_PATH, index=False)

print(f"\n✓ Final encoded dataset saved to: {FINAL_FILE_PATH}")

print(f"\nFinal dataframe information:")
print(f"  Shape: {df_encoded.shape}")
print(f"  Columns: {len(df_encoded.columns)}")
print(f"  Data types:\n{df_encoded.dtypes}")
print(f"\nMissing values:")
print(df_encoded.isnull().sum())

print(f"\nFirst few rows:")
print(df_encoded.head())

print("\n" + "=" * 80)
print("ENCODING PIPELINE COMPLETE")
print("=" * 80)


STEP 6: EXPORT FINAL ENCODED DATASET

✓ Final encoded dataset saved to: ../../data/final\encoded_unsupervised.csv

Final dataframe information:
  Shape: (32548, 22)
  Columns: 22
  Data types:
highest_education               int64
imd_band                        int64
age_band                        int64
num_of_prev_attempts          float64
studied_credits                 int64
date_registration             float64
module_presentation_length    float64
tma_cma_weighted_score        float64
total_submissions             float64
late_submissions_count        float64
avg_submission_delay          float64
code_module_BBB                 int64
code_module_CCC                 int64
code_module_DDD                 int64
code_module_EEE                 int64
code_module_FFF                 int64
code_module_GGG                 int64
code_presentation_2013J         int64
code_presentation_2014B         int64
code_presentation_2014J         int64
gender_M                        int64
disabili